## 절대 모멘텀
- 전년도(3개월, 6개월, 12개월)의 수정 주가와 전월의 수정주가를 이용하여 구매의 타이밍을 잡는 투자 전략 
- 구매 신호 -> (전월의 수정주가 / 전년도의 수정주가) - 1 값이 0보다 크고 무한대가 아닌 경우 

1. 파생변수 STD-YM 생성 -> index에서 년-월을 추출하여 대입 
2. STD-YM 별 마지막날의 데이터들을 모아서 month_last_df 데이터프레임을 생성 
3. 전월의 수정주가 파생변수 생성 -> 전월 수정 주가 대입 
4. 전년도의 수정주가 파생변수 생성 -> 전년도 수정 주가 대입
5. 구매 신호를 생성 
6. 원본의 데이터에서 구호 신호에 따른 거래 내역 생성 
7. 수익율 계산

In [1]:
import pandas as pd 
import numpy as np 
from datetime import datetime

In [3]:
df = pd.read_csv("../csv/AMZN.csv" , index_col='Date')
df.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [4]:
# index 데이터 시계열로 변환 
df.index = pd.to_datetime(df.index)

In [7]:
# index 데이터에서 년-월을 추출하여 STD-YM에 대입 
df.index.strftime('%Y-%m')

Index(['1997-05', '1997-05', '1997-05', '1997-05', '1997-05', '1997-05',
       '1997-05', '1997-05', '1997-05', '1997-05',
       ...
       '2019-06', '2019-06', '2019-06', '2019-06', '2019-06', '2019-06',
       '2019-06', '2019-06', '2019-06', '2019-06'],
      dtype='object', name='Date', length=5563)

In [ ]:
ym_list = []
for idx in df.index:
    ym = idx.strftime('%Y-%m')
    ym_list.append(ym)

ym_list

In [9]:
df['STD-YM'] = ym_list

In [12]:
df.loc[ '1997-06-25' : '1997-07-05',  ]

,Open,High,Low,Close,Adj Close,Volume,STD-YM
Date,,,,,,,
1997-06-25,1.526042,1.526042,1.489583,1.510417,1.510417,2106000,1997-06
1997-06-26,1.520833,1.520833,1.505208,1.510417,1.510417,3189600,1997-06
1997-06-27,1.515625,1.515625,1.479167,1.489583,1.489583,1188000,1997-06
1997-06-30,1.510417,1.598958,1.479167,1.541667,1.541667,2746800,1997-06
1997-07-01,1.541667,1.541667,1.510417,1.515625,1.515625,1292400,1997-07
1997-07-02,1.515625,1.593750,1.510417,1.588542,1.588542,3882000,1997-07
1997-07-03,1.598958,1.916667,1.593750,1.911458,1.911458,12577200,1997-07


In [ ]:
# 현재 행의 STD-YM과 다음 행의 STD-YM이 다른 경우 -> 월말
flag = df['STD-YM'] != df.shift(-1)['STD-YM']
df.loc[flag, ]

In [17]:
# 그룹화하고 마지막 데이터만 확인 
month_last_df = df.groupby('STD-YM').tail(1)

In [18]:
# 전월의 수정주가 , 전년도의 수정주가 컬럼을 생성 
month_last_df['BF-1M'] = month_last_df.shift(1)['Adj Close'].fillna(0)
month_last_df['BF-12M'] = month_last_df.shift(12)['Adj Close'].fillna(0)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_21076\4231060231.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  month_last_df['BF-1M'] = month_last_df.shift(1)['Adj Close'].fillna(0)
C:\Users\ekfla\AppData\Local\Temp\ipykernel_21076\4231060231.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  month_last_df['BF-12M'] = month_last_df.shift(12)['Adj Close'].fillna(0)


In [19]:
month_last_df.iloc[10: 15, ]

,Open,High,Low,Close,Adj Close,Volume,STD-YM,BF-1M,BF-12M
Date,,,,,,,,,
1998-03-31,7.114583,7.208333,6.979167,7.127600,7.127600,6565200,1998-03,6.416667,0.000000
1998-04-30,8.125000,8.166667,7.541667,7.645833,7.645833,22485600,1998-04,7.127600,0.000000
1998-05-29,7.156250,7.416667,7.125000,7.343750,7.343750,8641200,1998-05,7.645833,1.500000
1998-06-30,16.302084,16.916666,16.125000,16.625000,16.625000,21877200,1998-06,7.343750,1.541667
1998-07-31,19.083334,19.187500,18.187500,18.479166,18.479166,13440600,1998-07,16.625000,2.395833
